# 09. Performance Benchmarking: Conventional Python (Pandas) vs. PySpark
## ASBD Lab Exam — Analytics and Systems of Big Data

---
### 1. Objective and Benchmarking Methodology
Empirically benchmark execution times and scaling characteristics between single-threaded Python (Pandas/mlxtend) and distributed PySpark across:
1. **Data Loading:** Reading CSV files into DataFrames.
2. **Data Cleaning & Feature Engineering:** Imputation, length extraction, engagement calculation.
3. **GroupBy Aggregations:** Recipe performance summaries.
4. **FP-Growth Mining:** Frequent pattern mining at min_support=0.05.

**Scales Evaluated:** 10% (1,818 rows), 25% (4,545 rows), 50% (9,091 rows), 100% (18,182 rows).  
**Speedup Metric:**  
$$\text{Speedup} = \frac{\text{Execution Time}_{\text{Python}}}{\text{Execution Time}_{\text{PySpark}}}$$


In [ ]:
import sys
sys.path.append('..')
from src.benchmark import benchmark_pipeline
import pandas as pd

# Execute empirical benchmarking suite
df_bench = benchmark_pipeline(raw_csv='../dataset/dataset.csv', output_dir='../outputs/benchmarks')
df_bench


### 2. Speedup Analysis Table


In [ ]:
df_bench[['Scale_Pct', 'Records', 'Speedup_Load', 'Speedup_Clean', 'Speedup_Agg', 'Speedup_FPGrowth']]


### 3. Scaling Curves & Speedup Visualizations


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=150)
img1 = mpimg.imread('../outputs/benchmarks/benchmark_execution_times.png')
img2 = mpimg.imread('../outputs/benchmarks/benchmark_speedup_ratios.png')
axes[0].imshow(img1)
axes[0].axis('off')
axes[1].imshow(img2)
axes[1].axis('off')
plt.tight_layout()
plt.show()


### 4. Technical Conclusion on Big Data Tool Selection
- **The Small-to-Medium Data Paradox:** For datasets fitting comfortably in RAM (< 1–2 GB, e.g. 18,182 rows / 6 MB), Pandas outperforms PySpark because Pandas operates directly in native C-arrays without process isolation.
- **Spark Overhead Components:** PySpark incurs JVM invocation latency, Py4J socket IPC serialization, Catalyst query optimization, DAG stage generation, and task scheduling overhead (~100–300 ms fixed floor).
- **The Crossover Point:** PySpark becomes essential and dramatically outperforms Python when data exceeds single-machine RAM (e.g. > 10–50 GB), where Pandas crashes with `MemoryError` while Spark automatically handles out-of-core spilling, partition-level parallel execution, and multi-node cluster scaling.
